# Import thư viện và Load dữ liệu

In [1]:
import pandas as pd
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Tải dữ liệu đã làm sạch
df = pd.read_csv('../data/processed/movies_cleaned_soup.csv')

# Điền rỗng cho các giá trị NaN (đề phòng có lỗi trong quá trình lưu file)
df['soup'] = df['soup'].fillna('')

print(f"Sẵn sàng đưa {len(df)} bộ phim vào huấn luyện!")

Sẵn sàng đưa 9621 bộ phim vào huấn luyện!


# Vector hóa văn bản

In [2]:
print("⚙️ Đang tiến hành Vector hóa văn bản...")

# Khởi tạo TF-IDF Vectorizer
# max_features=5000: Chỉ lấy 5000 từ xuất hiện phổ biến và mang ý nghĩa phân loại cao nhất
# stop_words='english': Một lớp bảo vệ phụ đề phòng sót stop words ở bước trước
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)

# Dạy mô hình và chuyển đổi cột 'soup' thành Ma trận đặc trưng (Feature Matrix)
tfidf_matrix = tfidf.fit_transform(df['soup'])

print(f"✅ Hoàn tất! Kích thước Ma trận TF-IDF: {tfidf_matrix.shape}")
# Kích thước sẽ là (Số lượng phim, 5000)

⚙️ Đang tiến hành Vector hóa văn bản...
✅ Hoàn tất! Kích thước Ma trận TF-IDF: (9621, 5000)


# Tính toán khoảng cách

In [3]:
print("📐 Đang tính toán Ma trận tương đồng (Cosine Similarity)...")

# Tính toán điểm tương đồng giữa tất cả các phim với nhau
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print(f"✅ Hoàn tất! Kích thước Ma trận Cosine: {cosine_sim.shape}")
print("Mẫu dữ liệu điểm tương đồng của phim số 0 với các phim khác:")
print(cosine_sim[0][:10])

📐 Đang tính toán Ma trận tương đồng (Cosine Similarity)...
✅ Hoàn tất! Kích thước Ma trận Cosine: (9621, 9621)
Mẫu dữ liệu điểm tương đồng của phim số 0 với các phim khác:
[1.         0.02696397 0.03778794 0.02310226 0.00449469 0.00640913
 0.04517132 0.08438775 0.01009629 0.02268969]


# Testing

In [5]:
# Tạo một Series để tra cứu nhanh Index của phim dựa trên Title
indices = pd.Series(df.index, index=df['title']).drop_duplicates()

def get_recommendations(title, cosine_sim=cosine_sim):
    if title not in indices:
        return f"❌ Không tìm thấy phim '{title}' trong cơ sở dữ liệu."
    
    # Lấy index của bộ phim mà người dùng nhập
    idx = indices[title]
    
    # Lấy danh sách điểm tương đồng của phim này với tất cả phim khác
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # Sắp xếp danh sách theo điểm tương đồng giảm dần
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Lấy Top 5 phim giống nhất (bỏ qua phim ở vị trí 0 vì đó chính là nó)
    sim_scores = sim_scores[1:6]
    
    # Lấy index của Top 5 phim đó
    movie_indices = [i[0] for i in sim_scores]
    
    # Trả về tên của 5 bộ phim
    return df['title'].iloc[movie_indices].tolist()

# TEST THỬ NGAY:
print("🎬 Gợi ý cho phim 'Toy Story':")
print(get_recommendations('Toy Story (1995)'))

print("\n🎬 Gợi ý cho phim 'Dead Man Walking':")
print(get_recommendations('Dead Man Walking (1995)'))

🎬 Gợi ý cho phim 'Toy Story':
['Toy Story 2 (1999)', 'Toy Story 3 (2010)', 'Small Soldiers (1998)', 'Toys (1992)', 'Indian in the Cupboard, The (1995)']

🎬 Gợi ý cho phim 'Dead Man Walking':
['Into the Abyss (2011)', 'I Want to Live! (1958)', 'Life of David Gale, The (2003)', 'True Crime (1999)', 'Green Mile, The (1999)']
